In [2]:
import spateo as st
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
import numpy as np
import skimage
import cv2
from numba import njit
from joblib import Parallel, delayed
from tqdm import tqdm
import math
from skimage.segmentation import find_boundaries
import scanpy as sc
from skimage import measure
from shapely.geometry import LineString, MultiPolygon, Point, Polygon
from shapely.wkb import dumps
from anndata import AnnData

In [ ]:
gem_path = '/data/input/A02187D3/T202306120001/04.tissuecut/A02187D3.tissue.gem.gz'
ssDNA_0_path = '/data/work/01.Cellbin/Script/27/0904/Slice_27_ssDNA_0.tif'
ssDNA_1_path = '/data/work/01.Cellbin/Script/27/0904/Slice_27_ssDNA_1.tif'
save_path = '/data/work/01.Cellbin/02.Result/27/0904/'
ssDNA_path = '/data/work/01.Cellbin/data/A02187D3_adj.tif'
whole_geneExpression_save_path = '/data/work/01.Cellbin/02.Result/27/0904/Slice_27_whole_geneExpression.png'
raw_ssDNA_save_path = '/data/work/01.Cellbin/02.Result/27/0904/Slice_27_raw_ssDNA.png'
name = 'Slice_27'


In [4]:
def generate_geneExpression_png(
    gem_file_path: str, 
    save_path: str):

    gem_file = pd.read_csv(gem_file_path, sep='\t',comment='#')
    x, y = gem_file["x"].values, gem_file["y"].values
    shape = (x.max() + 1,y.max() + 1)
    X = csr_matrix((gem_file["MIDCount"].values, (x, y)), shape=shape, dtype=np.uint16)
    mtx = X.todense().astype(np.uint8)
    mtx_ = skimage.color.gray2rgb(np.array(mtx))
    plt.imsave(save_path, mtx_)
    return (x.max() + x.min())/2

In [5]:
def _expand_labels(
    labels: np.ndarray,
    distance: int,
    max_area: int,
    mask,
) -> np.ndarray:
    """Expand labels up to a certain distance, while ignoring labels that are
    above a certain size.

    Args:
        labels: Numpy array containing integer labels.
        distance: Distance to expand. Internally, this is used as the number
            of iterations of distance 1 dilations.
        max_area: Maximum area of each label.
        mask: Only expand within the provided mask.

    Returns:
        New label array with expanded labels.
    """
    masked_labels = labels[mask] if mask is not None else labels
    if (masked_labels > 0).all() or (masked_labels == 0).all():
        return labels

    @njit
    def _expand(X, areas, max_area, mask, start_i, end_i):
        expanded = X[start_i:end_i].copy()
        new_areas = np.zeros_like(areas)
        n_neighbors = 0
        neighbors = np.zeros(4, dtype=X.dtype)
        for i in range(start_i, end_i):
            for j in range(X.shape[1]):
                if X[i, j] > 0 or not mask[i, j]:
                    continue

                if i - 1 >= 0:
                    neighbors[n_neighbors] = X[i - 1, j]
                    n_neighbors += 1
                if i + 1 < X.shape[0]:
                    neighbors[n_neighbors] = X[i + 1, j]
                    n_neighbors += 1
                if j - 1 >= 0:
                    neighbors[n_neighbors] = X[i, j - 1]
                    n_neighbors += 1
                if j + 1 < X.shape[1]:
                    neighbors[n_neighbors] = X[i, j + 1]
                    n_neighbors += 1
                unique = np.unique(neighbors[:n_neighbors])
                unique_labels = unique[unique > 0]
                if len(unique_labels) == 1:
                    label = unique_labels[0]
                    if areas[label] < max_area:
                        expanded[i - start_i, j] = label
                        new_areas[label] += 1
                n_neighbors = 0
        return expanded, new_areas

    areas = np.bincount(labels.flatten())
    mask = np.ones(labels.shape, dtype=bool) if mask is None else mask
    step = math.ceil(labels.shape[0] / 119)
    expanded = labels.copy()
    with Parallel(n_jobs=119) as parallel:
        for _ in tqdm(range(distance), desc="Expanding"):
            new_areas = np.zeros_like(areas)
            subis = range(0, labels.shape[0], step)
            sublabels = []
            submasks = []
            for i in subis:
                sl = slice(max(0, i - 1), min(labels.shape[0], i + step + 1))
                sublabels.append(expanded[sl])
                submasks.append(mask[sl])
            for i, (_expanded, _new_areas) in zip(
                subis,
                parallel(
                    delayed(_expand)(
                        sl, areas, max_area, sm, int(i - 1 >= 0), sl.shape[0] - int(i + step + 1 < labels.shape[0])
                    )
                    for i, sl, sm in zip(subis, sublabels, submasks)
                ),
            ):
                expanded[i : i + step] = _expanded
                new_areas += _new_areas
            areas += new_areas

    return expanded

In [6]:
def cellbin_image_plot(adata_path, gem_path, tif_path, grey_path, save_path, thickness):
    adata = sc.read(adata_path)
    res_data = adata.layers['augmented_labels_expanded']
    labels = res_data.copy()
    bound = np.zeros_like(labels, dtype=np.uint8)
    thickness = 1
    for i in range(thickness):
        labels = np.where(bound == 0, labels, 0)
        bound_one = find_boundaries(labels, mode="inner").astype(np.uint8)
        bound += bound_one
    bound = bound * 255
    bound = skimage.color.gray2rgb(bound)
    plt.imsave(save_path + name + '_boundary.png', bound)
    
    gem = pd.read_csv(gem_path, sep = '\t', comment = '#')
    x_min = gem['x'].min()
    y_min = gem['y'].min()
    
    img = cv2.imread(grey_path)
    img = img[:,:,::-1] # transform image to rgb
    img_cut = img[x_min:,y_min:,:]
    plt.imsave(save_path + name + '_geneExpression.png', img_cut)
    
    img = cv2.imread(tif_path)
    img = img[:,:,::-1] # transform image to rgb
    img_cut = img[x_min:,y_min:,:]
    plt.imsave(save_path + name + '_ssDNA.png',img_cut)
    
    return

In [7]:
def read_bgi(
    path,
    segmentation_adata,
    labels_layer,
    
):

    binsize = None
    seg_binsize = 1
    label_column = None
    add_props = True
    version = 'stereo'
    seg_binsize = 1
    labels = None
    
    data = st.io.bgi.read_bgi_as_dataframe(path, label_column)
    n_columns = data.shape[1]

    uniq_gene = sorted(data["geneID"].unique())

    props = None

    binsize = 1
    shape = (data["x"].max(), data["y"].max())

    labels = segmentation_adata.layers[labels_layer]
    
    label_coords = get_coords_labels(labels)

    if labels_layer is not None:
        seg_binsize = 1
        x_min, y_min = (
            int(segmentation_adata.obs_names[0]) * seg_binsize,
            int(segmentation_adata.var_names[0]) * seg_binsize,
        )
        label_coords["x"] += x_min
        label_coords["y"] += y_min

    
    data = pd.merge(data, label_coords, on=["x", "y"], how="inner")

    props = get_label_props(labels)

    uniq_cell = sorted(data["label"].unique())
    shape = (len(uniq_cell), len(uniq_gene))
    cell_dict = dict(zip(uniq_cell, range(len(uniq_cell))))
    gene_dict = dict(zip(uniq_gene, range(len(uniq_gene))))
    x_ind = data["label"].map(cell_dict).astype(int).values
    y_ind = data["geneID"].map(gene_dict).astype(int).values

    X = csr_matrix((data["total"].values, (x_ind, y_ind)), shape=shape)
    layers = {}

    obs = pd.DataFrame(index=uniq_cell)
    var = pd.DataFrame(index=uniq_gene)
    adata = AnnData(X=X, obs=obs, var=var, layers=layers)
    if props is not None:
        ordered_props = props.loc[adata.obs_names]
        adata.obs["area"] = ordered_props["area"].values
        adata.obsm["spatial"] = ordered_props.filter(regex="centroid-").values
        # adata.obsm["contour"] = ordered_props["contour"].values
        adata.obsm["bbox"] = ordered_props.filter(regex="bbox-").values

    return adata

def get_coords_labels(labels: np.ndarray) -> pd.DataFrame:
    """Convert labels into sparse-format dataframe.

    Args:
        labels: cell segmentation labels matrix.

    Returns:
        A DataFrame of columns "x", "y", and "label". The coordinates are
        relative to the labels matrix.
    """
    nz = labels.nonzero()
    x, y = nz
    data = labels[nz]
    values = np.vstack((x, y, data)).T
    return pd.DataFrame(values, columns=["x", "y", "label"])


def contour_to_geo(contour):
    """Transfer contours to `shapely.geometry`"""
    n = contour.shape[0]
    if n >= 3:
        geo = Polygon(contour)
    elif n == 2:
        geo = LineString(contour)
    else:
        geo = Point(contour[0])
    geo = dumps(geo, hex=True)  # geometry object to hex
    return geo

def get_label_props(labels: np.ndarray) -> pd.DataFrame:
    """Measure properties of labeled cell regions.

    Args:
        labels: cell segmentation label matrix

    Returns:
        A dataframe with properties and contours indexed by label
    """

    def contour(mtx):
        """Get contours of a cell using `cv2.findContours`."""
        mtx = mtx.astype(np.uint8)
        contours = cv2.findContours(mtx, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
        # assert len(contours) == 1
        return contours[0].squeeze(1)

    props = measure.regionprops_table(
        labels, properties=("label", "area", "bbox", "centroid"), extra_properties=[contour]
    )
    props = pd.DataFrame(props)
    props["contour"] = props.apply(lambda x: x["contour"] + x[["bbox-0", "bbox-1"]].to_numpy(), axis=1)
    props["contour"] = props["contour"].apply(contour_to_geo)
    return props.set_index(props["label"].astype(str)).drop(columns="label")

In [14]:
mid = generate_geneExpression_png(gem_file_path = gem_path,
                                  save_path = whole_geneExpression_save_path)

In [15]:
ssDNA_tif = cv2.imread(ssDNA_path)
ssDNA_tif.shape

(15947, 21528, 3)

In [16]:
mid

10625.0

In [17]:
ssDNA_tif[int(mid):,:,:] = 0

In [18]:
cv2.imwrite(raw_ssDNA_save_path, ssDNA_tif)

True

In [19]:
ssDNA_0 = cv2.imread(ssDNA_0_path.replace('tif','jpg'), 0)
cv2.imwrite(ssDNA_0_path, ssDNA_0)

True

In [20]:
ssDNA_1 = cv2.imread(ssDNA_1_path.replace('tif','jpg'), 0)
cv2.imwrite(ssDNA_1_path, ssDNA_1)

True

In [21]:
adata_0 = st.io.read_bgi_agg(
    gem_path, ssDNA_0_path,
)

|-----> Constructing count matrices.
|-----> <insert> __type to uns in AnnData Object.
|-----> <insert> pp to uns in AnnData Object.
|-----> <insert> spatial to uns in AnnData Object.


In [22]:
st.cs.mask_nuclei_from_stain(adata_0)
st.pl.imshow(adata_0, 'stain_mask', save_show_or_return = 'save', save_kwargs={'path':save_path + '01.stain_mask_' + name})

|-----> <select> stain layer in AnnData Object
|-----> Constructing nuclei mask from staining image.
|-----> <insert> stain_mask to layers in AnnData Object.
|-----> <select> stain_mask layer in AnnData Object
Saving figure to /data/work/01.Cellbin/02.Result/27/0904/imshow_01.stain_mask_Slice_27.pdf...
Done


In [23]:
st.cs.find_peaks_from_mask(adata_0, 'stain', 1)
st.cs.watershed(adata_0, 'stain', 3, out_layer='watershed_labels')
st.pl.imshow(adata_0, 'watershed_labels', labels=True, save_show_or_return='save', save_kwargs={'path':save_path + '02.watershed_labels_' + name})

|-----> <select> stain_mask layer in AnnData Object
|-----> Finding peaks with minimum distance 1.
|-----> <insert> stain_distances to layers in AnnData Object.
|-----> <insert> stain_markers to layers in AnnData Object.
|-----> <select> stain layer in AnnData Object
|-----> <select> stain_mask layer in AnnData Object
|-----> <select> stain_markers layer in AnnData Object
|-----> Running Watershed.
|-----> <insert> watershed_labels to layers in AnnData Object.
|-----> <select> watershed_labels layer in AnnData Object
Saving figure to /data/work/01.Cellbin/02.Result/27/0904/imshow_02.watershed_labels_Slice_27.pdf...
Done


In [24]:
adata_1 = st.io.read_bgi_agg(
    gem_path, ssDNA_1_path,
)

|-----> Constructing count matrices.
|-----> <insert> __type to uns in AnnData Object.
|-----> <insert> pp to uns in AnnData Object.
|-----> <insert> spatial to uns in AnnData Object.


In [25]:
st.cs.stardist(adata_1, tilesize=-1, equalize=2.0, out_layer='stardist_labels')
st.pl.imshow(adata_1, 'stardist_labels', labels=True, save_show_or_return='save', save_kwargs={'path':save_path + '03.stardist_labels_' + name})

|-----> <select> stain layer in AnnData Object
|-----> Equalizing image with CLAHE.
|-----> Running StarDist with model 2D_versatile_fluo.
Found model '2D_versatile_fluo' for 'StarDist2D'.


2023-09-04 11:14:11.608354: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/software/python/lib/python3.8/site-packages/cv2/../../lib64:
2023-09-04 11:14:11.608396: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
/opt/software/python/lib/python3.8/site-packages/csbdeep/models/base_model.py:256: UserWarning:

skipping normalization step after prediction because number of input and output channels differ.



Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


2023-09-04 11:14:15.045905: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 20972371968 exceeds 10% of free system memory.
2023-09-04 11:14:16.600249: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 20972371968 exceeds 10% of free system memory.
2023-09-04 11:14:42.886256: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 20972371968 exceeds 10% of free system memory.


|-----> Fixing disconnected labels.
|-----> <insert> stardist_labels to layers in AnnData Object.
|-----> <select> stardist_labels layer in AnnData Object
Saving figure to /data/work/01.Cellbin/02.Result/27/0904/imshow_03.stardist_labels_Slice_27.pdf...
Done


In [26]:
adata = adata_0.copy()

In [27]:
adata

AnnData object with n_obs × n_vars = 10643 × 15364
    uns: '__type', 'pp', 'spatial'
    layers: 'stain', 'stain_mask', 'stain_distances', 'stain_markers', 'watershed_labels'

In [28]:
label_max = adata.layers['watershed_labels'].max()

In [29]:
label_max

91617

In [30]:
stardist_labels = np.where(adata_1.layers['stardist_labels']>0, adata_1.layers['stardist_labels'] + label_max + 1, 0)

In [31]:
adata.layers['augment_labels'] = adata.layers['watershed_labels'] + stardist_labels

In [32]:
st.cs.utils.filter_cell_labels_by_area(adata, 'augment_labels', 30)
adata

|-----> <select> augment_labels layer in AnnData Object
|-----> Cell number before filtering is 92219
|-----> <insert> augment_labels to layers in AnnData Object.
|-----> Cell number after filtering is 27273


AnnData object with n_obs × n_vars = 10643 × 15364
    uns: '__type', 'pp', 'spatial'
    layers: 'stain', 'stain_mask', 'stain_distances', 'stain_markers', 'watershed_labels', 'augment_labels'

In [33]:
st.pl.imshow(adata, 'augment_labels', labels=True, save_show_or_return='save', save_kwargs={'path':save_path + '04.augment_labels_' + name})

|-----> <select> augment_labels layer in AnnData Object
Saving figure to /data/work/01.Cellbin/02.Result/27/0904/imshow_04.augment_labels_Slice_27.pdf...
Done


In [34]:
adata.layers['augmented_labels_expanded'] = _expand_labels(adata.layers['augment_labels'], distance = 20, max_area = 30000, mask = None)
# st.cs.expand_labels(adata, 'augmented_labels', distance=20, max_area=12000)
st.pl.imshow(adata, 'augmented_labels_expanded', labels=True, save_show_or_return='save', save_kwargs={'path':save_path + '05.augmented_labels_expanded_' + name})

Expanding: 100%|██████████| 20/20 [02:09<00:00,  6.48s/it]


|-----> <select> augmented_labels_expanded layer in AnnData Object
Saving figure to /data/work/01.Cellbin/02.Result/27/0904/imshow_05.augmented_labels_expanded_Slice_27.pdf...
Done


In [35]:
adata.write(save_path + name + '_labeled.h5ad')

In [36]:
adata

AnnData object with n_obs × n_vars = 10643 × 15364
    uns: '__type', 'pp', 'spatial'
    layers: 'stain', 'stain_mask', 'stain_distances', 'stain_markers', 'watershed_labels', 'augment_labels', 'augmented_labels_expanded'

In [37]:
cell_adata = read_bgi(
    gem_path,
    segmentation_adata=adata,
    labels_layer='augmented_labels_expanded',
)
cell_adata

/opt/software/python/lib/python3.8/site-packages/anndata/_core/anndata.py:117: ImplicitModificationWarning:

Transforming to str index.



AnnData object with n_obs × n_vars = 27253 × 20573
    obs: 'area'
    obsm: 'spatial', 'bbox'

In [38]:
cell_adata.write(save_path + name + '.h5ad')

In [39]:
cellbin_image_plot(adata_path = save_path + name + '_labeled.h5ad',
                   gem_path = gem_path,
                   tif_path = ssDNA_path,
                   grey_path = whole_geneExpression_save_path,
                   save_path = save_path,
                   thickness = 1)